In [1]:
def get_the_right_cable(switch,Cables,absolute_sw_to_dev):
    cables = Cables[switch]
    higher_lengths = sorted([item for item in cables if isinstance(item.get('length'), 
                                    (int, float)) and item.get('length') > absolute_sw_to_dev], key=lambda x: x['length'])

    if higher_lengths:
        return higher_lengths[0]
    else:
        return {}

In [2]:
def calculate_cable_lengths(rack, LANs, Cables, colors_info):
    unit_to_cm = 4.45
    # the following are in units
    device_to_rackside = 6.75
    racktop_to_ceiling = 4.5 # assumed 20cm
    rack_to_rack = 4.5 # assumed from side to the adjacent side no spacing
    lans = dict()
    devices = dict()
    switches = dict()
    for dev in rack:
        dev_id = dev.split('_')[-1]
        device = '_'.join(dev.split('_')[:-1])
        if 'LAN' in device:
            lanname, switch = device.split('__')
            lans[dev] = dict()
            device_specs = [x for x in LANs[lanname]['switch'] if x['model'] == switch ][0]
            lans[dev] = device_specs.copy()
            lans[dev]['position'] = rack[dev]
            lans[dev]['id'] = dev_id
            lans[dev]['half_for_split'] = 0
            lans[dev]['full_no_split'] = 0
            lans[dev]['cables'] = dict()
            lans[dev]['minimum_cable_total_length'] = 0
            lans[dev]['actual_cable_total_length'] = 0
            if device not in switches:
                switches[device] = 0
            switches[device] +=1 
         
    for lan in lans:
        lanname = lan.split('__')[0]
        switch = lans[lan]['model']
        for dev in rack:
            dev_id = dev.split('_')[-1]
            device = '_'.join(dev.split('_')[:-1])
            if 'LAN' not in device :
                if colors_info[device][lanname]['count'] > 0:
                    if dev not in devices:
                        devices[dev] = dict()
                    devices[dev]['position'] = rack[dev]
                    devices[dev]['id'] = dev_id
                    no_ports = colors_info[device][lanname]['count']
                    speed_ratio = lans[lan]['speed'] / colors_info[device][lanname]['speed']
                    device_position = rack[dev]
                    # absolute cable lengeth in meters:
                    absolute_sw_to_dev = ((lans[lan]['position'] - device_position) + 2*device_to_rackside)*unit_to_cm/100
                    cable = get_the_right_cable(switch,Cables,absolute_sw_to_dev)
                    no_of_cables = colors_info[device][lanname]['count']
                    devices[dev][lan] = dict()
                    devices[dev][lan]['cable_count'] = colors_info[device][lanname]['count']
                    devices[dev][lan]['speed'] = colors_info[device][lanname]['speed']
                    if speed_ratio < 1:
                        lans[lan]['half_for_split'] +=1
                        devices[dev][lan]['half_for_split'] = 1
                    else:
                        lans[lan]['full_no_split'] += 1
                        devices[dev][lan]['full_no_split'] = 1
                    devices[dev][lan]['cables_count'] = no_of_cables
                    devices[dev][lan]['cable_type'] = cable
                    if cable['model'] not in lans[lan]['cables']:
                        lans[lan]['cables'][cable['model']] = 0                    
                    lans[lan]['cables'][cable['model']] += no_of_cables
                    lans[lan]['minimum_cable_total_length'] += absolute_sw_to_dev * no_of_cables
                    lans[lan]['actual_cable_total_length'] += cable['length'] * no_of_cables
    return (devices, lans, switches)

rack = {'LAN_1__IB+400_1': 41, 'FrontEnd_nodes_2': 40, 'GPU_nodes_3': 32, 'compute_nodes_4': 30, 'compute_nodes_5': 28, 'compute_nodes_6': 26, 'compute_nodes_7': 24, 'compute_nodes_8': 22}
lans, devices  = calculate_cable_lengths(rack, LANs, Cables, colors_info)
print('devices',devices)
print('lans',lans)

In [3]:
def sort_device_before_placement(all_items):
    lan_devices = sorted([item for item in all_items if 'LAN' in item['type']], key=lambda x: x['type'])
    other_devices = [item for item in all_items if 'LAN' not in item['type']]

    def sort_other(item):
        sort_key = [1] * len(lan_devices) + [item.get('type', '')] # Initialize with a lower priority and type for tie-breaking

        for i, lan_item in enumerate(lan_devices):
                for key, value in item.items():
                    if isinstance(value, dict):
                        sort_key[i] = -item[key].get('count', 0) # Higher count gets a higher (negative) priority
                        break # Move to the next LAN device after finding a match
        return tuple(sort_key)

    sorted_other_devices = sorted(other_devices, key=sort_other)
    return lan_devices + sorted_other_devices

all_servers = [{'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'LAN_3__IB+400'}, {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1': {'count':0, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 8, 'speed': 400}, 'type': 'GPU_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}, {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_1': {'count': 1, 'speed': 400}, 'LAN_2': {'count': 1, 'speed': 200}, 'LAN_3': {'count': 1, 'speed': 25}, 'LAN_4': {'count': 1, 'speed': 25}, 'LAN_5': {'count': 1, 'speed': 1}, 'LAN_6': {'count': 0, 'speed': 400}, 'type': 'compute_nodes'}]

sorted_servers = sort_device_before_placement(all_servers)
print(sorted_servers)

In [4]:
def old_sort_device_before_placement(item):
    if 'LAN' in item['type']:
        return (0, item['type'])  # Prioritize LAN, then sort alphabetically by type
    else:
        return (1, item['type'])  # Other types come later, maintain original order



In [5]:
10//4

2

In [114]:
import numpy as np
from collections import Counter
import functools
import time
from functools import cache, lru_cache, wraps
global_rack_signature = dict()

def is_rack_stable(rack_weight, rack_height_mm, rack_cg_height_mm, servers, rack_depth_mm):
    """
    Checks if the rack configuration is likely stable based on the combined
    vertical center of gravity.
    """
    total_weight = rack_weight + sum(s[1] for s in servers)
    if total_weight == 0:
        return True

    combined_vertical_cg = (rack_weight * rack_cg_height_mm +
                             sum(s[1] * s[0] for s in servers)) / total_weight

    stability_threshold_fraction = 0.7  # Adjust as needed
    return combined_vertical_cg <= rack_height_mm * stability_threshold_fraction



import collections
from functools import wraps

def to_hashable(obj):
    """Convert objects to hashable forms while preserving structure."""
    if isinstance(obj, Counter):
        # For Counter, convert to tuple of items
        return ('__counter__', tuple(sorted(obj.items())))
    elif isinstance(obj, dict):
        # For dicts, convert to tuple of sorted items
        return ('__dict__', tuple(sorted((k, to_hashable(v)) for k, v in obj.items())))
    elif isinstance(obj, list):
        return ('__list__', tuple(to_hashable(item) for item in obj))
    elif isinstance(obj, tuple):
        return ('__tuple__', tuple(to_hashable(item) for item in obj)) 
    elif isinstance(obj, set):
        return ('__set__', frozenset(to_hashable(item) for item in obj))
    return obj

def from_hashable(obj):
    """Convert back from hashable form to original objects."""
    if isinstance(obj, tuple) and len(obj) == 2:
        type_tag, value = obj
        if type_tag == '__counter__':
            # Rebuild Counter from items
            return Counter(dict(value))
        elif type_tag == '__dict__':
            # Rebuild dict
            return {k: from_hashable(v) for k, v in value}
        elif type_tag == '__list__':
            # Rebuild list
            return [from_hashable(item) for item in value]
        elif type_tag == '__tuple__':
            # Keep as tuple but convert contents
            return tuple(from_hashable(item) for item in value)
        elif type_tag == '__set__':
            # Rebuild set
            return {from_hashable(item) for item in value}
    
    # If it's a regular tuple (not tagged), process its elements
    if isinstance(obj, tuple):
        return tuple(from_hashable(item) for item in obj)
    
    return obj

def hashable_cache(func):
    """
    Decorator that makes function arguments hashable for caching,
    then converts them back to their original types when calling the function.
    """
    @functools.lru_cache(maxsize=None)
    def cached_wrapper(*hashable_args, **hashable_kwargs):
        # Convert the hashable arguments back to their original types
        restored_args = tuple(from_hashable(arg) for arg in hashable_args)
        restored_kwargs = {k: from_hashable(v) for k, v in hashable_kwargs.items()}
        
        # Call the original function with restored arguments
        return func(*restored_args, **restored_kwargs)
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Convert arguments to hashable versions
        hashable_args = tuple(to_hashable(arg) for arg in args)
        hashable_kwargs = {k: to_hashable(v) for k, v in kwargs.items()}
        
        # Call the cached version
        return cached_wrapper(*hashable_args, **hashable_kwargs)
    
    # Add cache control methods
    wrapper.cache_clear = cached_wrapper.cache_clear
    wrapper.cache_info = cached_wrapper.cache_info
    
    return wrapper

# Example usage - rename this to @hashable_args if that's what your code expects
def hashable_args(func):
    return hashable_cache(func)

# Test function to demonstrate usage
#@hashable_args
def find_stable_positions_greedy_complex(rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
                                         servers_to_place, LANs, Cables, colors_info, prioritize_top=False):
    """
    A greedy approach to find stable server positions for various server types inside one rack
    """
    global global_rack_signature
    
    tohash = dict()
    for key,value in enumerate(servers_to_place):
        tohash[str(key)+str(value)] = 1
    signature = frozenset(tohash)
    signature = tuple(sorted(servers_to_place.items()))
    if signature in global_rack_signature:
        return global_rack_signature[signature]

    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2

    
    all_servers = []
    for server_type, count in servers_to_place.items():
        specs = colors_info.get(server_type)
        if not specs:
            if 'LAN' in server_type:
                lan , sw_model = server_type.split('__')
                
                specs = [x for x in LANs[lan]['switch'] if x['model'] == sw_model][0]
                
            else:
                print(f"Warning: Specifications not found for server type '{server_type}'. Skipping.")
                continue
           
                
        specs['type'] = server_type
        for _ in range(count):
                all_servers.append(specs)
    

    if not all_servers:
        return {}, {}, {}  # Return empty dict for empty Counter
    all_servers = sort_device_before_placement(all_servers)
    
    lan_id = 0
    
    periority_lan = 'NA'
    workinglans = [x['type'] for x in all_servers if 'LAN' in x['type']]
    lan_loops = {}
    for lan in workinglans:
        loops =  len(all_servers) - 2
        lan_loops[lan] = {'loops':loops,'best_cable_lengths':float('inf'), 'best_position':0,'referrenced_lan':lan+'_10000000'}

    excluded_swap = []
    best_placement = ()
    best_cable_lengths = float('inf')
    best_position = 0
    if len(workinglans) == 0:
        workinglans = ['na']
    
    for lan in workinglans:
        if lan == 'na':
            loops = 0
        else:
            loops = lan_loops[lan]['loops']
        best_cable_lengths = float('inf')
        periority_lan = lan
        best_position = 0
        swap_pos = 0
        loops += 1
        original_allservers = list(all_servers)
        for _ in range(loops):
            if not prioritize_top:
                placed_servers_info = []
                occupied_u = [False] * rack_height_u
                for server in all_servers:
                    server_height_u = server['height']
                    server_weight_kg = server['weight']
                    server_cg_offset = (server_height_u * u_height_mm) / 2
                    placed = False
                    for i in range(rack_height_u):
                        if not occupied_u[i]:
                            start_u = i
                            server_base_height = start_u * u_height_mm
                            server_cg = server_base_height + server_cg_offset
                            can_place = True
                            for u_check in range(start_u, start_u + server_height_u):
                                if u_check >= rack_height_u or occupied_u[u_check]:
                                    can_place = False
                                    break
                            if can_place:
                                temp_positions = [(p['cg'], p['weight']) for p in placed_servers_info] + [(server_cg, server_weight_kg)]
                                if is_rack_stable(rack_weight_kg, rack_height_mm, rack_cg_height_mm, temp_positions, rack_depth_mm):
                                    placed_servers_info.append({'cg': server_cg, 'weight': server_weight_kg, 'type': server['type'], 'height': server['height'], 'start_u': start_u})
                                    for u in range(start_u, start_u + server_height_u):
                                        if u < rack_height_u:
                                            occupied_u[u] = True
                                    placed = True
                                    break
                    if not placed:
                        return None, None, None

                final_placement = {}
                for i, server_info in enumerate(placed_servers_info):
                    final_placement[f"{server_info['type']}_{i+1}"] = server_info['start_u'] + 1
            else:
                placed_servers_info = []
                occupied_u = [False] * rack_height_u
                for server in all_servers:
                    server_height_u = server['height']
                    server_weight_kg = server['weight']
                    server_cg_offset = (server_height_u * u_height_mm) / 2
                    placed = False
                    for i in range(rack_height_u - 1, -1, -1):
                        if not occupied_u[i]:
                            start_u = i
                            server_base_height = start_u * u_height_mm
                            server_cg = server_base_height + server_cg_offset
                            can_place = True
                            for u_check in range(start_u, start_u + server_height_u):
                                if u_check >= rack_height_u or occupied_u[u_check]:
                                    can_place = False
                                    break
                            if can_place:
                                temp_positions = [(p['cg'], p['weight']) for p in placed_servers_info] + [(server_cg, server_weight_kg)]
                                if is_rack_stable(rack_weight_kg, rack_height_mm, rack_cg_height_mm, temp_positions, rack_depth_mm):
                                    placed_servers_info.append({'cg': server_cg, 'weight': server_weight_kg, 'type': server['type'], 'height': server['height'], 'start_u': start_u})
                                    for u in range(start_u, start_u + server_height_u):
                                        if u < rack_height_u:
                                            occupied_u[u] = True
                                    placed = True
                                    break
                    if not placed:
                        return {}, {}, {}
                    
               
                final_placement = {}
                sorted_servers = sorted(placed_servers_info, key=lambda x: x['start_u'], reverse=True)
                occupied_map = [False] * rack_height_u
                placed_index = 1
                for server in sorted_servers:
                    start_u = server['start_u']
                    server_type = server['type']
                    server_height = server['height']
                    for u in range(start_u, -1, -1):
                        can_place_here = True
                        for check_u in range(u, u + server_height):
                            if check_u >= rack_height_u or occupied_map[check_u]:
                                can_place_here = False
                                break
                        if can_place_here:
                            final_placement[f"{server_type}_{placed_index}"] = u + 1
                            for occupy_u in range(u, u + server_height):
                                if occupy_u < rack_height_u:
                                    occupied_map[occupy_u] = True
                            placed_index += 1
                            break
                        else:
                            # Fallback to the initially found stable position
                            final_placement[f"{server_type}_{placed_index}"] = start_u + 1
                            for occupy_u in range(start_u, start_u + server_height):
                                if occupy_u < rack_height_u:
                                    occupied_map[occupy_u] = True
                            placed_index += 1
            
            devices, lans, rackswitches = calculate_cable_lengths(final_placement,LANs, Cables,colors_info)
            
            if 'LAN' in periority_lan:
                for t_lan in lans:
                    if periority_lan in t_lan:
                        referrenced_lan = t_lan
                        break
                best_flag = 0
                check_preceding_cable_length = 0
                for old_lan in lan_loops:
                    for t_lan in lans:
                        if old_lan in t_lan:
                            old_current_referrenced_lan = t_lan
                            break
                    old_best_cable_lengths = lan_loops[old_lan]['best_cable_lengths']
                    check_preceding_cable_length =  lan_loops[old_lan]['best_cable_lengths'] - lans[old_current_referrenced_lan]['actual_cable_total_length']
                    if check_preceding_cable_length >  0:
                        old_preferred_lan = old_lan
                        best_flag = 1
                    if check_preceding_cable_length < 0:
                        break

                if best_flag and check_preceding_cable_length == 0 : 
                    best_flag = 0
                    for old_lan in lan_loops:
                        for t_lan in lans:
                            if old_lan in t_lan:
                                old_current_referrenced_lan = t_lan
                            break
                        old_referrenced_lan = lan_loops[old_lan]['referrenced_lan']
                        check_current_position = final_placement[old_current_referrenced_lan] - lan_loops[old_lan]['best_position']
                        if check_current_position < 0  :
                            best_flag = 0
                            break
                        if check_current_position > 0:
                            best_flag = 2
                        else:
                            if best_flag == 1:
                                best_flag = 0
                        
                if best_flag == 1:
                    best_placement = (final_placement.copy(), devices.copy(),lans.copy(), rackswitches.copy())
                    #print('best_placement',best_placement[0])
                    #print('servers_to_place',servers_to_place)
                    #sssss
                    for old_lan in lan_loops: 
                        for t_lan in lans:
                            if old_lan in t_lan:
                                old_current_referrenced_lan = t_lan
                            break
                        lan_loops[old_lan]['best_position'] = final_placement[old_current_referrenced_lan]
                        lan_loops[old_lan]['best_cable_lengths'] = lans[old_current_referrenced_lan]['actual_cable_total_length']
                        lan_loops[old_lan]['referrenced_lan'] = lan_loops[old_lan]['referrenced_lan']
                        #print('new placement',best_cable_lengths,'with',best_position,'for',referrenced_lan)
                #print('still',best_cable_lengths, 'best_position',best_position)
                all_servers[swap_pos], all_servers[swap_pos + 1] = all_servers[swap_pos+1], all_servers[swap_pos]
                swap_pos += 1
            else:
                #print('no lans there')
                best_placement = (final_placement.copy(), {},{},{})
                break
    if len(best_placement) < 4:
        print(all_servers)
        print('=============================')
        print(best_placement) 
    global_rack_signature[signature] = best_placement
    return best_placement
    


# --- Example Usage ---
rack_height_u = 42
rack_weight_kg = 114.55
rack_width_mm = 600
rack_depth_mm = 1200

all_racks_config = [
    Counter({'LAN_1__IB+400':1,'compute_nodes': 5, 'GPU_nodes': 1, 'FrontEnd_nodes': 1}),
    Counter({'FrontEnd_nodes': 12, 'GPU_nodes': 1}),
    Counter({'compute_nodes': 5, 'GPU_nodes': 1, 'FrontEnd_nodes': 1}),
    Counter({'FrontEnd_nodes': 15, 'compute_nodes': 5}),
    Counter({'compute_nodes': 5, 'GPU_nodes': 1, 'FrontEnd_nodes': 1}),
    Counter({'storage_nodes': 8}),
    Counter({'storage_nodes': 15}),
    Counter({'storage_nodes': 7, 'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter({'GPU_nodes': 1}),
    Counter(),
    Counter(),
]


colors_info = {
        'compute_nodes': {'count': 20, 'wattage': 1600, 'height': 2, 'weight':30, 'LAN_1':{'count':1,'speed':400},
                                                                           'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1,'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':0, 'speed':400},
                      },
        'GPU_nodes': {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':8, 'speed':400},
                      },
        'storage_nodes': {'count': 30, 'wattage': 1250, 'height': 2, 'weight':20, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':3,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                          'LAN_6':{'count':0, 'speed':400},
                         },
        'FrontEnd_nodes': {'count': 30, 'wattage': 750, 'height': 1, 'weight':9, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':0,'speed':200},
                                                                            'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                         'LAN_6':{'count':0, 'speed':400},
                          },
                   
    }
LANs = { 'LAN_1':{'type':'400gbsNDR','topology': 'halfports_spine_leaf',
                  'switch':[{'model':'IB+400','ports':64,'speed':400, 'height':2,'wattage':2000, 'weight':20},
                         {'model':'IB_800','ports':64,'speed':800, 'height':2,'wattage':2000, 'weight':20},
                           ]},
              'LAN_2':{'type':'400GbpsEth', 'topology': 'uplinks_spine_leaf',
'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
                 {'model':'Z_6xxx','ports':64,'speed':200, 'height':1,'wattage':2000, 'uplink_count':4, 'uplink_speed':400,'weight':20},
        ]},
              'LAN_3':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300, 'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_4':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300,'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':2, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_5':{'type':'1gbps','topology': 'uplinks_spine_leaf',
    'switch':[{'model':'SN_24','ports':24,'speed':1, 'height':1,'wattage':200, 'uplink_count':2, 'uplink_speed':25, 'weight':2},
             ]},
              'LAN_6':{'type':'400GbpsEth','topology':'rails',
    'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
             ]},
           }
Cables = {'IB+400':[{'model':'1.5m_400IB_copper','server_port_speed':400,'split':1,'length':1.5},
                                           {'model':'3m_400IB_copper','server_port_speed':400,'split':1,'length':3},
                                         {'model':'5m_400IB_fiber','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400IB_fiber','server_port_speed':400,'split':1,'length':7},
                                      { 'model':'10m_400IB_fiber','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400IB_fiber','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400sIB_copper','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400sIB_fiber','server_port_speed':200,'split':2,'length':5},
                                      { 'model':'7m_400sIB_fiber','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400sIB_fiber','server_port_speed':200,'split':2,'length':10},
                                     {  'model':'20m_400sIB_fiber','server_port_speed':200,'split':2,'length':20},
                            ],
                            
          'IB_800':[{'model':'1.5m_800IB_copper','server_port_speed':800,'split':1,'length':1.5},
                                       {'model':'3m_800IB_copper','server_port_speed':800,'split':1,'length':3},
                                      { 'model':'8m_800IB_fiber','server_port_speed':800,'split':1,'length':5},
                                      { 'model':'7m_800IB_fiber','server_port_speed':800,'split':1,'length':7},
                                      { 'model':'10m_800IB_fiber','server_port_speed':800,'split':1,'length':10},
                                       {'model':'20m_800IB_fiber','server_port_speed':800,'split':1,'length':20},
                                      { 'model':'3m_800sIB_copper','server_port_speed':400,'split':2,'length':3},
                                     {  'model':'5m_800sIB_fiber','server_port_speed':400,'split':2,'length':5},
                                      { 'model':'7m_800sIB_fiber','server_port_speed':400,'split':2,'length':7},
                                      { 'model':'10m_800sIB_fiber','server_port_speed':400,'split':2,'length':10},
                                      { 'model':'20m_800sIB_fiber','server_port_speed':400,'split':2,'length':20},
                   ],
          'S_xx64':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                     {  'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'S_xx32':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                       {'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'SN_24':[{'model':'1.5m_1_copper','server_port_speed':1,'split':1,'length':1.5},
                                      { 'model':'3m_1_coppe','server_port_speed':1,'split':1,'length':3},
                                       {'model':'5m_1_copper','server_port_speed':1,'split':1,'length':5},
                                      { 'model':'7m_1_copper','server_port_speed':1,'split':1,'length':7},
                                      { 'model':'10m_1_copper','server_port_speed':1,'split':1,'length':10},
                                      { 'model':'20m_1_copper','server_port_speed':1,'split':1,'length':20}, 
                                      { 'model':'30m_1_copper','server_port_speed':1,'split':1,'length':30},
                                       {'model':'40m_1_copper','server_port_speed':1,'split':1,'length':40},
                  ],
          'Z_9xxx':[{'model':'1.5m_400_copper_eth','server_port_speed':400,'split':1,'length':1.5},
                                       {'model':'3m_400_copper_eth','server_port_speed':400,'split':1,'length':3},
                                      { 'model':'5m_400_fiber_eth','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber_eth','server_port_speed':400,'split':1,'length':7},
                                     {  'model':'10m_400_fiber_eth','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber_eth','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400s_copper_eth','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber_eth','server_port_speed':200,'split':2,'length':5},
                                     {  'model':'7m_400s_fiber_eth','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber_eth','server_port_speed':200,'split':2,'length':10},
                                      { 'model':'20m_400s_fiber_eth','server_port_speed':200,'split':2,'length':20},
                   ],
         }         
            
          

results = []

for rack_config in all_racks_config:
    if rack_config:
        #print(f"Processing rack with config: {rack_config}")
        # Bottom-up placement
        stable_placement_bottom, devices, lans = find_stable_positions_greedy_complex(
            rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            rack_config, LANs, Cables, colors_info, prioritize_top=True
        )
        print(f" Top-biased Placement: {stable_placement_bottom}")

        # Top-biased placement
        stable_placement_top_biased, devices, lans = find_stable_positions_greedy_complex(
            rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            rack_config, LANs, Cables, colors_info, prioritize_top=False
        )
        print(f" Bottom-up Placement: {stable_placement_top_biased}")
        print("-" * 30)

print("Processing complete.")

In [49]:
def get_rack_layouts(distributions,Cables,LANs, colors_info):
    rack_height_u = 42
    rack_weight_kg = 114.55
    rack_width_mm = 600
    rack_depth_mm = 1200
    distributions_info = dict()
    for i, rack_config in enumerate(distributions):
        if rack_config:
            switches = dict()
            # Top-biased placement
            
            stable_placement , devices, lans, rackswitches = find_stable_positions_greedy_complex(
                rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
                rack_config, LANs, Cables, colors_info, prioritize_top=True
            )
            distributions_info[i] = dict({'rack_config':rack_config, 'stable_placement': stable_placement ,'lan_info':lans,
                                               'device_info':devices, 'switches':rackswitches})
           
            #print(f" Top-biased Placement: {stable_placement_bottom}")

            
             # Bottom-up placement
            #stable_placement_top_biased = find_stable_positions_greedy_complex(
            #    rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            #    rack_config,Cables, LANs, colors_info, prioritize_top=false
            #)
            #print(f"   Bottom-up Placement:{stable_placement_top_biased}")
            #print("-" * 30)
    
    #print("Processing rack layouts Complete.")
    return  distributions_info

In [40]:
import numpy as np
import os
import time
import multiprocessing as mp
from collections import Counter
from itertools import product, permutations
from math import ceil
import logging
import random

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def calculate_box_wattage(box, colors_wattage):
    return sum(box.get(color, 0) * colors_wattage.get(color.split('__')[-1], 0) for color in box)

def calculate_box_height(box, colors_height):
    return sum(box.get(color, 0) * colors_height.get(color.split('__')[-1], 0) for color in box)

def check_box_limits(box, colors_wattage, colors_height, max_box_wattage, max_box_height):
    box_wattage = calculate_box_wattage(box, colors_wattage)
    box_height = calculate_box_height(box, colors_height)
    return box_wattage <= max_box_wattage and box_height <= max_box_height
def get_total_cable_lengths(distributions_info):
    global_minimum_length = 0
    global_actual_length = 0
    global_switches = dict()
    cables = dict()
    
    for rack_info in distributions_info:
        if 'lan_info' in distributions_info[rack_info]:
            for lan in distributions_info[rack_info]['lan_info']:
                global_minimum_length += distributions_info[rack_info]['lan_info'][lan]['minimum_cable_total_length']
                global_actual_length += distributions_info[rack_info]['lan_info'][lan]['actual_cable_total_length']
                for cable in distributions_info[rack_info]['lan_info'][lan]['cables']:
                    if cable not in cables:
                        cables[cable] = 0
                    cables[cable] += distributions_info[rack_info]['lan_info'][lan]['cables'][cable]
        if 'switches' in distributions_info[rack_info]:
            for sw_lan in distributions_info[rack_info]['switches']:
                if sw_lan not in global_switches:
                    global_switches[sw_lan] = 0
                global_switches[sw_lan] += distributions_info[rack_info]['switches'][sw_lan]
    
    distributions_info['cables'] = {'global_minimum_length':global_minimum_length, 'global_actual_length':global_actual_length,
                                   'cables':cables }
    distributions_info['switches'] = global_switches.copy()

    return distributions_info

def display_distribution_filled_only(distributions_info, colors_wattage, colors_height, total_balls, balls_placed_overall):
    print(f"\nValid Distribution Found (Iterative Greedy - Filled Racks Only):")
    current_total_balls = sum(balls_placed_overall.values())
    total_wattage = 0
    total_height = 0
    total_count = 0
    filled_boxes = []
    for i, info in enumerate(distributions_info):
        if 'stable_placement' in distributions_info[info]:
            total_count += len(distributions_info[info]['stable_placement'])
       
        if 'rack_config' in distributions_info[info]:
            rack_config = distributions_info[info]['rack_config']
            #if len(rack_config) == 1 and 'LAN' in str(rack_config):
            #    continue
            box_contents = ", ".join(f"{count} {color}" for color, count in rack_config.items())
            box_wattage = calculate_box_wattage(rack_config, colors_wattage)
            box_height = calculate_box_height(rack_config, colors_height)
            total_wattage += box_wattage
            total_height += box_height
            ball_count = sum(rack_config.values())
            filled_boxes.append(f"\x1b[1m -------------------------Rack {i+1} info------------------------------\x1b[0m")
            filled_boxes.append(f"\x1b[1;4m-->Rack {i+1}:\x1b[0m \n {box_contents} (Wattage: {box_wattage}, Height: {box_height}, count:{ball_count})")
            filled_boxes.append(f"-->\x1b[1;4mdevice placement: in the rack \x1b[0m \n {distributions_info[info]['stable_placement']}")
            filled_boxes.append(f"-->\x1b[1;4maggregated LAN info: \x1b[0m \n {distributions_info[info]['lan_info']}")
            #filled_boxes.append(f"-->\x1b[1;4mDevice detailed info: \x1b[0m \n {distributions_info[info]['device_info']}")

    filled_boxes.append(f"\x1b[1;4m-------------------------Total distribution info------------------------------ \x1b[0m \n {distributions_info['cables']} \n {distributions_info['switches']}")
    if filled_boxes:
        for box_info in filled_boxes:
            print('rack_info', box_info)
        print(f"  Total Wattage: {total_wattage}, Total Height: {total_height}, Total Balls Distributed:{total_count} {current_total_balls}")
    else:
        print(f"  No racks were filled. Total Balls Distributed: {current_total_balls}")
    if current_total_balls != total_balls:
        print(f"  WARNING: Total balls distributed ({current_total_balls}) does not match the expected total ({total_balls})!")
    

def is_switch_relevent(rack, switch_info, colors_info):
    alllans = set()
    allnodes = ""
    include_lan = set()
    execlude_lan = set()
    for device in rack:
        if 'LAN' in device:
            switch = device.split('__')[-1]
            lan = device.split('__')[0]
            alllans.add(lan)
    is_there_device = 0
    for device in rack:
        if 'LAN' not in device:
            for lan in alllans:
                if colors_info[device][lan]['count'] > 0:
                    include_lan.add(lan)
    execlude_lan = alllans - include_lan
    return list(execlude_lan)

def get_distribution_signature(distribution):
        #start_time = time.time()
        filled_boxes_signature = tuple(sorted(tuple(sorted(box.items())) for box in distribution if box))
        #end_time = time.time()
        #print(f"Update: the signing the distribution is calcualted in {end_time - start_time:.6f} seconds")
        return filled_boxes_signature

def format_time_difference_compact(seconds):
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    days, hours = divmod(hours, 24)

    parts = []
    if days > 0:
        parts.append(f"{int(days)}d")
    if hours > 0:
        parts.append(f"{int(hours)}h")
    if minutes > 0:
        parts.append(f"{int(minutes)}m")
    parts.append(f"{seconds:.3f}s")  # Show seconds always, with 3 decimal places for compactness

    return " ".join(parts) or "0.000s" 
    
def find_valid_distributions_iterative_greedy_adaptive(LANs, Cables, switch_info, colors_info, initial_num_boxes, max_box_wattage, max_box_height, total_balls):
    colors = list(colors_info.keys())
    colors_wattage = {color: colors_info[color]['wattage'] for color in colors}
    colors_wattage.update({switch_info[lan][0]['model']: switch_info[lan][0]['wattage'] for lan in list(switch_info.keys())})
    
    colors_height = {color: colors_info[color]['height'] for color in colors}
    colors_height.update({switch_info[lan][0]['model']: switch_info[lan][0]['height'] for lan in list(switch_info.keys())})
    
    max_possible_boxes = total_balls
    valid_distributions = []
    seen_distributions = set()

    

    num_boxes_to_try = list(range(1, initial_num_boxes + max_possible_boxes + 1))
    random.shuffle(num_boxes_to_try)
    lan_len = len(LANs)
    switch_scenarios = []
    switch_scenarios.append("switch_per_1_rack")
    #switch_scenarios.append("switch_per_2_Racks")
    #switch_scenarios.append("switch_per_3_Racks")
    #switch_scenarios.append("switch_per_4_Racks")
    #switch_scenarios.append("switch_per_5_Racks")
    #switch_scenarios.append("switch_per_6_Racks")
    swtich_index = 0
    start_time = time.time()
    current_switches = switch_info.copy()
    swcounter = 0 
    iter_counter = 1
    switch_to_add = dict()
    for lan in switch_info:
        switch_to_add[lan] = lan+ '__' + switch_info[lan][0]['model']
    best_minimum_length = float('inf')
    best_actual_length = float('inf')
    best_distributions = []
    best_min_devices = float('inf')
    best_LAN_check = []
    for _ in LANs:
        best_LAN_check.append(float('inf'))
    while iter_counter:
        iter_counter += 1
        for sw_scenario in switch_scenarios:
            for num_boxes in num_boxes_to_try:
                distributions = [Counter() for _ in range(num_boxes)]
                #if sw_scenario.split('_')[2] == '1':
                #    for box in distributions:
                #        for sw in switch_to_add:
                #            box[sw] = 1
                            
                        
                #else:
                #    break
                stub_rack = distributions[0]
                balls_placed = {color: 0 for color in colors}
                box_index = 0
                while box_index < num_boxes and any(balls_placed[color] < colors_info[color]['count'] for color in colors):
                    current_box = distributions[box_index]

                    remaining_balls = {c: colors_info[c]['count'] - balls_placed[c] for c in colors}
                    possible_colors = [c for c in colors if remaining_balls[c] > 0]

                    if not possible_colors:
                        box_index += 1
                        continue
                    
                    scenarios = []
                    for color in possible_colors:
                        #scenarios.append(("single_color", color))
                        for iddc in range(0,101):
                            scenarios.append(("partial_color_"+str(iddc), color))
                        #scenarios.append(("mixed_fill_initial", color))
                    
                    random.shuffle(scenarios) # Try scenarios in a random order for each box
            
                   
                    applied_scenario = False
                    for scenario_type, main_color in scenarios:
                        temp_box = current_box.copy()
                        temp_balls_placed = balls_placed.copy()
                        temp_remaining_balls = remaining_balls.copy()
                        
                        

                        if scenario_type.startswith("partial_color"):
                           
                            percentage = float(int(scenario_type.split('_')[2])/100)
                            add_amount = min(ceil(colors_info[main_color]['count'] * percentage), temp_remaining_balls[main_color])
                            
                            can_add = True
                            cycles = add_amount
                            recent_sw = []
                            while cycles:
                                if not recent_sw:
                                    for lan in colors_info[main_color]:
                                        if 'LAN' in lan:
                                            if lan not in str(temp_box):
                                                if int(colors_info[main_color][lan]['count']) > 0:
                                                    for to_add in switch_to_add:
                                                        if not box_index % int(sw_scenario.split('_')[2]):
                                                            if lan in to_add:
                                                                temp_box[switch_to_add[to_add]] = 1
                                                                recent_sw.append(switch_to_add[to_add])
                                if check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                   calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[main_color] <= max_box_wattage and \
                                   calculate_box_height(temp_box, colors_height) + colors_height[main_color] <= max_box_height and \
                                   temp_remaining_balls[main_color] > 0:
                                    temp_box[main_color] += 1
                                    temp_balls_placed[main_color] += 1
                                    temp_remaining_balls[main_color] -= 1
                                    cycles -= 1
                                else:
                                    #print(temp_box)
                                    #print(recent_sw)
                                    #if recent_sw:
                                    #    sw = recent_sw.pop()
                                    #    temp_box.pop(sw)
                                    cycles = 0
                                   
                            #if any(count > 0 for color, count in temp_remaining_balls.items() if color != main_color):
                            #others = [x for x in list(temp_remaining_balls.keys()) if x != main_color]
                            #if temp_remaining_balls.get(main_color, 0) > 0: 
                            #    others.append(main_color)
                            iteratecolor = 1
                            removed_sw = []
                            cycled = 1
                            twice_state = 2
                            ball_added = 1
                            while twice_state:
                                twice_state -=1
                                others = [x for x in list(temp_remaining_balls.keys()) if x != main_color]
                                if temp_remaining_balls.get(main_color, 0) > 0: 
                                    others.append(main_color)
                                #print('oooo',others)
                                #print('before',temp_box)
                                recent_sw = []
                                for other_color in others:
                                    if not recent_sw:
                                        for lan in colors_info[other_color]:
                                            if 'LAN' in lan:
                                                if lan not in str(temp_box):
                                                    if int(colors_info[other_color][lan]['count']) > 0:
                                                        for to_add in switch_to_add:
                                                            if not box_index % int(sw_scenario.split('_')[2]):
                                                                if lan in to_add:
                                                                    temp_box[switch_to_add[to_add]] = 1
                                                                    recent_sw.append(switch_to_add[to_add])


                                    #if temp_remaining_balls.get(other_color, 0) > 0:
                                    #print('checking again',temp_box)
                                    #print('temp_remaining_balls[other_color] > 0',temp_remaining_balls[other_color] > 0)
                                    #print('box_limits',check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height))
                                    #print('wattage',calculate_box_wattage(temp_box, colors_wattage) , colors_wattage.get(other_color, 0), max_box_wattage)
                                    #print('height',calculate_box_height(temp_box, colors_height) + colors_height.get(other_color, 0) <= max_box_height)

                                    if temp_remaining_balls[other_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                            calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(other_color, 0) <= max_box_wattage and \
                                            calculate_box_height(temp_box, colors_height) + colors_height.get(other_color, 0) <= max_box_height:
                                        temp_box[other_color] = temp_box.get(other_color, 0) + 1
                                        temp_balls_placed[other_color] = temp_balls_placed.get(other_color, 0) + 1
                                        temp_remaining_balls[other_color] -= 1
                                        iteratecolor = 1
                                        ball_added = 1
                                        #print(temp_box)

                                    else:
                                        while recent_sw:
                                            can_pop = 1
                                            sw_lan = recent_sw.pop()
                                            for device in temp_box:
                                                if 'LAN' not in device:
                                                    if int(colors_info[device][sw_lan.split('__')[0]]['count']) > 0:
                                                        can_pop = 0
                                                        break
                                            if can_pop:
                                                temp_box.pop(sw_lan)
                                                #twice_state = 3
                                                #print('popped')
                                                break
                                        
                                        if twice_state >= 1 and ball_added:
                                            twice_state = 3
                                            ball_added = 0
                                            #print('ball was added so ts = 3')
                                            
                                        elif twice_state == 2:
                                            twice_state = 0
                                       
                                        #print('eelse',temp_box, ball_added)
                                        #print(twice_state)
                                        
                                        

                            #print('ball_added',ball_added)            
                            
                                    
                                           
                                #print('t----',temp_box)
                           
                            if (can_add or add_amount > 0) and temp_box != current_box:
                                distributions[box_index] = temp_box
                                balls_placed.update(temp_balls_placed)
                                applied_scenario = True
                                break
                            h4444444
                        elif scenario_type == "mixed_fill_initial":
                            iteratecolor = 1
                            removed_sw = []
                            while iteratecolor:
                                if temp_remaining_balls[main_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                        calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[main_color] <= max_box_wattage and \
                                        calculate_box_height(temp_box, colors_height) + colors_height[main_color] <= max_box_height:
                                    temp_box[main_color] += 1
                                    temp_balls_placed[main_color] += 1
                                    temp_remaining_balls[main_color] -= 1
                                    remvoed_sw = []
                                else:
                                    is_removed = 0
                                    for idd in range(lan_len,0,-1):
                                        if 'LAN_'+str(idd) in str(temp_box) and colors_info[main_color]['LAN_'+str(idd)]['count'] == 0:
                                            removed_sw.append({key: value for key, value in temp_box.items() if 'LAN_'+str(idd) in key})
                                            temp_box = {key: value for key, value in temp_box.items() if 'LAN_'+str(idd) not in key}
                                            is_removed = 1
                                            break
                                    if not is_removed and iteratecolor:
                                        for removedsw in removed_sw:
                                            key = next(iter(removedsw.keys()))
                                            value = next(iter(removedsw.values()))
                                            temp_box[key] = value
                                        iteratecolor = 0

                            other_colors = [c for c in possible_colors if c != main_color]
                            random.shuffle(other_colors)
                            iteratecolor = 1
                            removed_sw = []
                            while iteratecolor:
                                if check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                        calculate_box_wattage(temp_box, colors_wattage) < max_box_wattage and \
                                        calculate_box_height(temp_box, colors_height) < max_box_height and \
                                        any(temp_remaining_balls[c] > 0 for c in other_colors):
                                    added_other = False
                                    for other_color in other_colors:
                                        if temp_remaining_balls[other_color] > 0 and \
                                           check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                           calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[other_color] <= max_box_wattage and \
                                           calculate_box_height(temp_box, colors_height) + colors_height[other_color] <= max_box_height:
                                            temp_box[other_color] += 1
                                            temp_balls_placed[other_color] += 1
                                            temp_remaining_balls[other_color] -= 1
                                            added_other = True
                                            removed_sw = []
                                            break
                                    if not added_other:
                                        is_removed = 0
                                        for idd in range(lan_len,0,-1):
                                            lanfound = 0
                                            if 'LAN_'+str(idd) not in str(temp_box):
                                                    continue
                                            for other_color in other_colors+[main_color]:
                                                if  colors_info[other_color]['LAN_'+str(idd)]['count'] > 0:
                                                    lanfound = 1
                                                    break
                                            if lanfound:
                                                continue
                                            if not lanfound:
                                                removed_sw.append({key: value for key, value in temp_box.items() if 'LAN_'+str(idd) in key})
                                                temp_box = {key: value for key, value in temp_box.items() if 'LAN_'+str(idd) not in key}
                                                is_removed = 1
                                                break
                                        if not is_removed and iteratecolor:
                                            for removedsw in removed_sw:
                                                key = next(iter(removedsw.keys()))
                                                value = next(iter(removedsw.values()))
                                                temp_box[key] = value
                                            iteratecolor = 0
                                       
                            if temp_box != current_box:
                                distributions[box_index] = temp_box
                                balls_placed.update(temp_balls_placed)
                                applied_scenario = True
                                break
                    advance_box = 0        
                    if applied_scenario or not possible_colors:
                        advance_box = 1
                    elif not any(remaining_balls.values()):
                        advance_box = 1
                    if advance_box:
                        execlude_switch = is_switch_relevent(temp_box,switch_info,colors_info)
                        #print(f" before {temp_box}")
                        if execlude_switch:
                            advanece_box = 0
                            for lan in execlude_switch:
                                temp_box = Counter({k: v for k, v in temp_box.items() if lan not in k})
                                #print(f" after {temp_box}")
                                distributions[box_index] = temp_box

                        box_index += 1
                
                if sum(balls_placed.values()) == total_balls:
                    remove_indices = []
                    for i,rack in enumerate(distributions):
                        remove_rack = 1
                        for item in rack:
                            if 'LAN' not in str(item):
                                remove_rack = 0
                                break
                        if remove_rack:
                            remove_indices.append(i)
                    for i in reversed(remove_indices):  
                        del distributions[i]
                    
                    signature = get_distribution_signature(distributions)
                    if signature not in seen_distributions:
                        seen_distributions.add(signature)
                        distributions_info = get_rack_layouts(distributions,Cables, LANs, colors_info)
                        distributions_info = get_total_cable_lengths(distributions_info)
                        total_devices = 0
                        for rack in distributions:
                                total_devices  += sum(rack.values())
                        LAN_check = [float('inf')] * len(LANs)
                        for id in range(len(LANs)):
                            for key in distributions_info['switches']:
                                if key.startswith('LAN_'+str(id)):
                                    LAN_check[id] = distributions_info['switches'][key]
                                    break
                        actual_length_check = distributions_info['cables']['global_actual_length']
                        total_length_check = sum(LAN_check)
                        if (actual_length_check < best_actual_length) or \
                        (actual_length_check == best_actual_length and LAN_check[0] < best_LAN_check[0]) or \
                         (actual_length_check == best_actual_length and LAN_check[0] == best_LAN_check[0] and total_devices < best_min_devices) :
                            best_min_devices = total_devices
                            best_LAN_check = LAN_check.copy()
                            best_distributions = distributions_info
                            best_actual_length = distributions_info['cables']['global_actual_length']
                            display_distribution_filled_only(distributions_info, colors_wattage, colors_height, total_balls, balls_placed)
                            valid_distributions.append(list(distributions))
                        # Potentially break here if you only need one solution
                    max_iter = 1000000
                    if iter_counter > max_iter:
                        return seen_distributions, valid_distributions, iter_counter
                    if iter_counter/500 == iter_counter //500:
                        #print(f"passing the {iter_counter} of {max_iter}")
                        end_time = time.time()
                        print(f"Update: the seen/validated distribution No {len(seen_distributions)} is calcualted in {end_time - start_time:.6f} sec after passing {iter_counter} iterations and rack cash {len(global_rack_signature)}",end='\r', flush=True)
                        start_time = time.time()
                    iter_counter += 1
                    #end_time = time.time()
                    #print(f"Update: the seen/validated distribution No{len(seen_distributions)} is calcualted in {end_time - start_time:.6f} seconds")
                    #start_time = time.time()
    return seen_distributions, valid_distributions, iter_counter

In [115]:
def main_iterative_greedy_adaptive_with_height():
    colors_info = {
        'compute_nodes': {'count':416, 'wattage': 1600, 'height': 2, 'weight':30, 'LAN_1':{'count':1,'speed':400},
                                                                           'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1,'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':0, 'speed':400},
                      },
        'GPU_nodes': {'count':15, 'wattage': 11000, 'height': 8, 'weight': 15, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                            'LAN_6':{'count':8, 'speed':400},
                      },
        'storage_nodes': {'count': 86, 'wattage': 1250, 'height': 2, 'weight':20, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':2,'speed':200},
                                                                           'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                          'LAN_6':{'count':0, 'speed':400},
                         },
        'FrontEnd_nodes': {'count': 30, 'wattage': 750, 'height': 1, 'weight':9, 'LAN_1':{'count':0,'speed':400},
                                                                            'LAN_2':{'count':1,'speed':200},
                                                                            'LAN_3':{'count':1, 'speed':25},
                                                                           'LAN_4':{'count':1, 'speed':25},
                                                                           'LAN_5':{'count':1, 'speed':1},
                                                                         'LAN_6':{'count':0, 'speed':400},
                          },
                   
    }
    LANs = { 'LAN_1':{'type':'400gbsNDR','topology': 'halfports_spine_leaf',
                  'switch':[{'model':'IB+400','ports':64,'speed':400, 'height':2,'wattage':2000, 'weight':20},
                         {'model':'IB_800','ports':64,'speed':800, 'height':2,'wattage':2000, 'weight':20},
                           ]},
              'LAN_2':{'type':'400GbpsEth', 'topology': 'uplinks_spine_leaf',
'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
                 {'model':'Z_6xxx','ports':64,'speed':200, 'height':1,'wattage':2000, 'uplink_count':4, 'uplink_speed':400,'weight':20},
        ]},
              'LAN_3':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300, 'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':4, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_4':{'type':'25gbps','topology': 'uplinks_spine_leaf',
        'switch':[{'model':'S_xx64','ports':64,'speed':25, 'height':2,'wattage':300,'uplink_count':4, 'uplink_speed':100,'weight':2},
                {'model':'S_xx32','ports':32,'speed':25, 'height':1,'wattage':200,'uplink_count':2, 'uplink_speed':100,'weight':2},
                 ]},
              'LAN_5':{'type':'1gbps','topology': 'uplinks_spine_leaf',
    'switch':[{'model':'SN_24','ports':24,'speed':1, 'height':1,'wattage':200, 'uplink_count':2, 'uplink_speed':25, 'weight':2},
             ]},
              'LAN_6':{'type':'400GbpsEth','topology':'rails',
    'switch':[{'model':'Z_9xxx','ports':64,'speed':400, 'height':2,'wattage':2200, 'uplink_count':4, 'uplink_speed':800,'weight':20},
             ]},
           }
    Cables = {'IB+400':[{'model':'1.5m_400IB_copper','server_port_speed':400,'split':1,'length':1.5},
                                           {'model':'3m_400IB_copper','server_port_speed':400,'split':1,'length':3},
                                         {'model':'5m_400IB_fiber','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400IB_fiber','server_port_speed':400,'split':1,'length':7},
                                      { 'model':'10m_400IB_fiber','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400IB_fiber','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400sIB_copper','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400sIB_fiber','server_port_speed':200,'split':2,'length':5},
                                      { 'model':'7m_400sIB_fiber','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400sIB_fiber','server_port_speed':200,'split':2,'length':10},
                                     {  'model':'20m_400sIB_fiber','server_port_speed':200,'split':2,'length':20},
                            ],
                            
          'IB_800':[{'model':'1.5m_800IB_copper','server_port_speed':800,'split':1,'length':1.5},
                                       {'model':'3m_800IB_copper','server_port_speed':800,'split':1,'length':3},
                                      { 'model':'8m_800IB_fiber','server_port_speed':800,'split':1,'length':5},
                                      { 'model':'7m_800IB_fiber','server_port_speed':800,'split':1,'length':7},
                                      { 'model':'10m_800IB_fiber','server_port_speed':800,'split':1,'length':10},
                                       {'model':'20m_800IB_fiber','server_port_speed':800,'split':1,'length':20},
                                      { 'model':'3m_800sIB_copper','server_port_speed':400,'split':2,'length':3},
                                     {  'model':'5m_800sIB_fiber','server_port_speed':400,'split':2,'length':5},
                                      { 'model':'7m_800sIB_fiber','server_port_speed':400,'split':2,'length':7},
                                      { 'model':'10m_800sIB_fiber','server_port_speed':400,'split':2,'length':10},
                                      { 'model':'20m_800sIB_fiber','server_port_speed':400,'split':2,'length':20},
                   ],
          'S_xx64':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                     {  'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'S_xx32':[{'model':'1.5m_25_copper','server_port_speed':25,'split':1,'length':1.5},
                                       {'model':'3m_25_copper','server_port_speed':25,'split':1,'length':3},
                                       {'model':'5m_25_fiber','server_port_speed':25,'split':1,'length':5},
                                       {'model':'7m_25_fiber','server_port_speed':25,'split':1,'length':7},
                                       {'model':'10m_25_fiber','server_port_speed':25,'split':1,'length':10},
                                       {'model':'20m_25_fiber','server_port_speed':25,'split':1,'length':20},                                       
                   ],
          'SN_24':[{'model':'1.5m_1_copper','server_port_speed':1,'split':1,'length':1.5},
                                      { 'model':'3m_1_coppe','server_port_speed':1,'split':1,'length':3},
                                       {'model':'5m_1_copper','server_port_speed':1,'split':1,'length':5},
                                      { 'model':'7m_1_copper','server_port_speed':1,'split':1,'length':7},
                                      { 'model':'10m_1_copper','server_port_speed':1,'split':1,'length':10},
                                      { 'model':'20m_1_copper','server_port_speed':1,'split':1,'length':20}, 
                                      { 'model':'30m_1_copper','server_port_speed':1,'split':1,'length':30},
                                       {'model':'40m_1_copper','server_port_speed':1,'split':1,'length':40},
                  ],
          'Z_9xxx':[{'model':'1.5m_400_copper_eth','server_port_speed':400,'split':1,'length':1.5},
                                       {'model':'3m_400_copper_eth','server_port_speed':400,'split':1,'length':3},
                                      { 'model':'5m_400_fiber_eth','server_port_speed':400,'split':1,'length':5},
                                      { 'model':'7m_400_fiber_eth','server_port_speed':400,'split':1,'length':7},
                                     {  'model':'10m_400_fiber_eth','server_port_speed':400,'split':1,'length':10},
                                      { 'model':'20m_400_fiber_eth','server_port_speed':400,'split':1,'length':20},
                                      { 'model':'3m_400s_copper_eth','server_port_speed':200,'split':2,'length':3},
                                     {  'model':'5m_400s_fiber_eth','server_port_speed':200,'split':2,'length':5},
                                     {  'model':'7m_400s_fiber_eth','server_port_speed':200,'split':2,'length':7},
                                      { 'model':'10m_400s_fiber_eth','server_port_speed':200,'split':2,'length':10},
                                      { 'model':'20m_400s_fiber_eth','server_port_speed':200,'split':2,'length':20},
                   ],
         }         
            
          
    
    num_devices = sum(info['count'] for info in colors_info.values())
    max_needed_wattage = sum(colors_info[color]['wattage'] * colors_info[color]['count'] for color in colors_info)
    max_box_wattage = 20000
    max_box_height = 42
    # arrange the list of lANs that are found int eh color_info
    current_lans = []
    str_nodes= str(colors_info)
    for lan in LANs:
        if lan in str_nodes:
            current_lans.append(lan)
   
    #create a dictionary same like the color_info which include the LANs various switches
    switch_info = dict()
    for lan in current_lans:
        switch_info[lan] = LANs[lan]['switch'].copy()
            
    #share this dictionary with the find_valid_distributions function
    initial_num_boxes = ceil(max_needed_wattage / max_box_wattage) + 2

    print(f"Distributing {num_devices} balls into initially {initial_num_boxes} boxes which consumes {max_needed_wattage} watts (Iterative Greedy Adaptive with Height):")

    start_time = time.time()
    seen_distributions, valid_distributions, iterations = find_valid_distributions_iterative_greedy_adaptive(
        LANs,Cables, switch_info,  colors_info, initial_num_boxes, max_box_wattage, max_box_height, num_devices
    )
    end_time = time.time()
    elapsedtime = format_time_difference_compact(end_time-start_time)

    print(f"\nSummary: Found {len(seen_distributions)} unique distribution(s) with {len(valid_distributions)} times of optimum length after {iterations} iterations tries in {elapsedtime} seconds")

if __name__ == "__main__":
    #global_rack_signature = dict()
    main_iterative_greedy_adaptive_with_height()
    print('fffffffffffffffffffffffffffffffffffffff')

Distributing 547 balls into initially 51 boxes which consumes 960600 watts (Iterative Greedy Adaptive with Height):

Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_1__IB+400, 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 9 compute_nodes (Wattage: 19400, Height: 27, count:14)
rack_info -->device placement: in the rack  
 {'LAN_4__S_xx64_1': 41, 'LAN_5__SN_24_2': 40, 'compute_nodes_3': 38, 'compute_nodes_4': 36, 'compute_nodes_5': 34, 'compute_nodes_6': 32, 'compute_nodes_7': 30, 'compute_nodes_8': 28, 'compute_nodes_9': 26, 'compute_nodes_10': 24, 'compute_nodes_11': 22, 'LAN_3__S_xx64_12': 20, 'LAN_1__IB+400_13': 18, 'LAN_2__Z_9xxx_14': 16}
rack_info -->aggregated LAN info:  
 {'LAN_4__S_xx64_1': {'model': 'S_xx64', 'ports': 64, 'speed': 25, 'height': 2, 'wattage': 300, 'uplink_count': 4, 'uplink_speed': 100, 'weight': 2, 'type': 'LAN_4__S_xx


Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 8 FrontEnd_nodes, 1 LAN_1__IB+400, 3 compute_nodes, 3 storage_nodes (Wattage: 19550, Height: 29, count:19)
rack_info -->device placement: in the rack  
 {'LAN_1__IB+400_1': 41, 'LAN_2__Z_9xxx_2': 39, 'LAN_3__S_xx64_3': 37, 'LAN_4__S_xx64_4': 35, 'LAN_5__SN_24_5': 34, 'compute_nodes_6': 32, 'compute_nodes_7': 30, 'compute_nodes_8': 28, 'FrontEnd_nodes_9': 27, 'FrontEnd_nodes_10': 26, 'FrontEnd_nodes_11': 25, 'FrontEnd_nodes_12': 24, 'FrontEnd_nodes_13': 23, 'FrontEnd_nodes_14': 22, 'FrontEnd_nodes_15': 21, 'FrontEnd_nodes_16': 20, 'storage_nodes_17': 18, 'storage_nodes_18': 16, 'storage_nodes_19': 14}
rack_info -->aggregated LAN info:  
 {'LAN_1__IB+400_1': {'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'L


Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 13 storage_nodes, 1 FrontEnd_nodes (Wattage: 20000, Height: 34, count:18)
rack_info -->device placement: in the rack  
 {'LAN_3__S_xx64_1': 41, 'LAN_2__Z_9xxx_2': 39, 'LAN_4__S_xx64_3': 37, 'LAN_5__SN_24_4': 36, 'FrontEnd_nodes_5': 35, 'storage_nodes_6': 33, 'storage_nodes_7': 31, 'storage_nodes_8': 29, 'storage_nodes_9': 27, 'storage_nodes_10': 25, 'storage_nodes_11': 23, 'storage_nodes_12': 21, 'storage_nodes_13': 19, 'storage_nodes_14': 17, 'storage_nodes_15': 15, 'storage_nodes_16': 13, 'storage_nodes_17': 11, 'storage_nodes_18': 9}
rack_info -->aggregated LAN info:  
 {'LAN_3__S_xx64_1': {'model': 'S_xx64', 'ports': 64, 'speed': 25, 'height': 2, 'wattage': 300, 'uplink_count': 4, 'uplink_speed': 100, 'weight': 2, 'type': 'LAN_3__S_xx64', 'position': 4


Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 22 FrontEnd_nodes (Wattage: 19500, Height: 29, count:26)
rack_info -->device placement: in the rack  
 {'FrontEnd_nodes_1': 42, 'FrontEnd_nodes_2': 41, 'LAN_5__SN_24_3': 40, 'FrontEnd_nodes_4': 39, 'FrontEnd_nodes_5': 38, 'FrontEnd_nodes_6': 37, 'FrontEnd_nodes_7': 36, 'FrontEnd_nodes_8': 35, 'FrontEnd_nodes_9': 34, 'FrontEnd_nodes_10': 33, 'FrontEnd_nodes_11': 32, 'FrontEnd_nodes_12': 31, 'FrontEnd_nodes_13': 30, 'FrontEnd_nodes_14': 29, 'FrontEnd_nodes_15': 28, 'FrontEnd_nodes_16': 27, 'FrontEnd_nodes_17': 26, 'FrontEnd_nodes_18': 25, 'FrontEnd_nodes_19': 24, 'FrontEnd_nodes_20': 23, 'FrontEnd_nodes_21': 22, 'FrontEnd_nodes_22': 21, 'FrontEnd_nodes_23': 20, 'LAN_2__Z_9xxx_24': 18, 'LAN_3__S_xx64_25': 16, 'LAN_4__S_xx64_26': 14}
rack_info -->aggregated LA


Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_1__IB+400, 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 9 compute_nodes (Wattage: 19400, Height: 27, count:14)
rack_info -->device placement: in the rack  
 {'LAN_4__S_xx64_1': 41, 'LAN_5__SN_24_2': 40, 'compute_nodes_3': 38, 'compute_nodes_4': 36, 'compute_nodes_5': 34, 'compute_nodes_6': 32, 'compute_nodes_7': 30, 'compute_nodes_8': 28, 'compute_nodes_9': 26, 'compute_nodes_10': 24, 'compute_nodes_11': 22, 'LAN_3__S_xx64_12': 20, 'LAN_1__IB+400_13': 18, 'LAN_2__Z_9xxx_14': 16}
rack_info -->aggregated LAN info:  
 {'LAN_4__S_xx64_1': {'model': 'S_xx64', 'ports': 64, 'speed': 25, 'height': 2, 'wattage': 300, 'uplink_count': 4, 'uplink_speed': 100, 'weight': 2, 'type': 'LAN_4__S_xx64', 'position': 41, 'id': '1', 'half_for_split': 0, 'full_no_split': 9, 'cables': {'1.5m_25_copper': 9}, 'minimum_c

...

Update: the seen/validated distribution No 3481 is calcualted in 11.526247 sec after passing 9500 iterations and rack cash 107
Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 7 storage_nodes, 1 LAN_1__IB+400, 2 compute_nodes, 4 FrontEnd_nodes (Wattage: 19950, Height: 31, count:18)
rack_info -->device placement: in the rack  
 {'LAN_1__IB+400_1': 41, 'LAN_2__Z_9xxx_2': 39, 'LAN_3__S_xx64_3': 37, 'LAN_4__S_xx64_4': 35, 'LAN_5__SN_24_5': 34, 'compute_nodes_6': 32, 'compute_nodes_7': 30, 'FrontEnd_nodes_8': 29, 'FrontEnd_nodes_9': 28, 'FrontEnd_nodes_10': 27, 'FrontEnd_nodes_11': 26, 'storage_nodes_12': 24, 'storage_nodes_13': 22, 'storage_nodes_14': 20, 'storage_nodes_15': 18, 'storage_nodes_16': 16, 'storage_nodes_17': 14, 'storage_nodes_18': 12}
rack_info -->aggregated LAN info:  
 {'LAN_1__IB+400_1': {'m

Update: the seen/validated distribution No 4066 is calcualted in 11.621866 sec after passing 12000 iterations and rack cash 109
Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 3 FrontEnd_nodes, 1 LAN_1__IB+400, 1 compute_nodes, 1 GPU_nodes, 1 LAN_6__Z_9xxx (Wattage: 22050, Height: 24, count:11)
rack_info -->device placement: in the rack  
 {'LAN_1__IB+400_1': 41, 'LAN_2__Z_9xxx_2': 39, 'LAN_3__S_xx64_3': 37, 'LAN_4__S_xx64_4': 35, 'LAN_5__SN_24_5': 34, 'LAN_6__Z_9xxx_6': 32, 'compute_nodes_7': 30, 'FrontEnd_nodes_8': 29, 'FrontEnd_nodes_9': 28, 'FrontEnd_nodes_10': 27, 'GPU_nodes_11': 19}
rack_info -->aggregated LAN info:  
 {'LAN_1__IB+400_1': {'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'LAN_1__IB+400', 'position': 41, 'id': '1', 'half_for_split': 0

Update: the seen/validated distribution No 12036 is calcualted in 11.482672 sec after passing 61500 iterations and rack cash 118
Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 10 FrontEnd_nodes, 1 LAN_1__IB+400, 3 compute_nodes, 2 storage_nodes (Wattage: 19800, Height: 29, count:20)
rack_info -->device placement: in the rack  
 {'LAN_1__IB+400_1': 41, 'LAN_2__Z_9xxx_2': 39, 'LAN_3__S_xx64_3': 37, 'LAN_4__S_xx64_4': 35, 'LAN_5__SN_24_5': 34, 'compute_nodes_6': 32, 'compute_nodes_7': 30, 'compute_nodes_8': 28, 'FrontEnd_nodes_9': 27, 'FrontEnd_nodes_10': 26, 'FrontEnd_nodes_11': 25, 'FrontEnd_nodes_12': 24, 'FrontEnd_nodes_13': 23, 'FrontEnd_nodes_14': 22, 'FrontEnd_nodes_15': 21, 'FrontEnd_nodes_16': 20, 'FrontEnd_nodes_17': 19, 'FrontEnd_nodes_18': 18, 'storage_nodes_19': 16, 'storage_nodes_20': 14}
rac

Update: the seen/validated distribution No 53905 is calcualted in 11.449480 sec after passing 785500 iterations and rack cash 125
Valid Distribution Found (Iterative Greedy - Filled Racks Only):
rack_info  -------------------------Rack 1 info------------------------------
rack_info -->Rack 1: 
 1 LAN_2__Z_9xxx, 1 LAN_3__S_xx64, 1 LAN_4__S_xx64, 1 LAN_5__SN_24, 1 LAN_6__Z_9xxx, 1 GPU_nodes, 1 LAN_1__IB+400, 1 compute_nodes (Wattage: 19800, Height: 21, count:8)
rack_info -->device placement: in the rack  
 {'LAN_1__IB+400_1': 41, 'LAN_2__Z_9xxx_2': 39, 'LAN_3__S_xx64_3': 37, 'LAN_4__S_xx64_4': 35, 'LAN_5__SN_24_5': 34, 'LAN_6__Z_9xxx_6': 32, 'compute_nodes_7': 30, 'GPU_nodes_8': 22}
rack_info -->aggregated LAN info:  
 {'LAN_1__IB+400_1': {'model': 'IB+400', 'ports': 64, 'speed': 400, 'height': 2, 'wattage': 2000, 'weight': 20, 'type': 'LAN_1__IB+400', 'position': 41, 'id': '1', 'half_for_split': 0, 'full_no_split': 1, 'cables': {'1.5m_400IB_copper': 1}, 'minimum_cable_total_length': 1.0

Update: the seen/validated distribution No 61632 is calcualted in 11.435927 sec after passing 1000000 iterations and rack cash 126
Summary: Found 61632 unique distribution(s) with 13 times of optimum length after 1000001 iterations tries in 6h 22m 1.082s seconds
fffffffffffffffffffffffffffffffffffffff


In [79]:
dict((('LAN_1__IB+400', 1), ('LAN_2__Z_9xxx', 1), ('LAN_3__S_xx64', 1), ('LAN_4__S_xx64', 1), ('LAN_5__SN_24', 1), ('compute_nodes', 9)))

{'LAN_1__IB+400': 1,
 'LAN_2__Z_9xxx': 1,
 'LAN_3__S_xx64': 1,
 'LAN_4__S_xx64': 1,
 'LAN_5__SN_24': 1,
 'compute_nodes': 9}